# TSP com AIMA (Artificial Intelligence: A Modern Approach)

**Tarefa 1 — ICSI30 (Sistemas Inteligentes)**

Integração com biblioteca AIMA: usando as classes e algoritmos da implementação official do livro de Russell & Norvig.

## Vantagens desta abordagem
- Código testado e usado em cursos mundo afora
- Direto com livro de referência (4ª ed.)
- Documentação inline explicando algoritmos
- Fácil extensão e customização

In [ ]:
import sys
import os

# Adicionar aima_lib ao path
aima_path = os.path.join(os.getcwd(), '..', 'aima_lib')
if aima_path not in sys.path:
    sys.path.insert(0, aima_path)

# Importar de AIMA
from aima.search import Problem
from aima.utils import *

import numpy as np
import matplotlib.pyplot as plt
import random
from typing import List, Tuple
import math

random.seed(42)
np.random.seed(42)

print("✓ AIMA carregado com sucesso")
print(f"✓ Path: {aima_path}")

## 1. Definir TSProblem herdando de Problem (AIMA)

A classe `Problem` do AIMA é a base para definir qualquer problema de busca:
- `initial`: estado inicial
- `actions(state)`: ações possíveis
- `result(state, action)`: novo estado após ação
- `goal_test(state)`: verifica se é objetivo
- `path_cost(c, s1, a, s2)`: custo do caminho
- `value(state)`: valor heurístico (para otimização)

In [ ]:
class TSPProblem(Problem):
    """
    Traveling Salesman Problem utilizando AIMA.
    
    Estado: lista de cidades em ordem [c0, c1, ..., cn]
    Objetivo: minimizar distância total (problema de otimização)
    """
    
    def __init__(self, cities: np.ndarray, initial_state=None):
        """
        Args:
            cities: array (n, 2) com coordenadas das cidades
            initial_state: tour inicial (aleatório se None)
        """
        self.cities = cities
        self.n_cities = len(cities)
        
        # Pré-computar distâncias
        self.distances = self._compute_distances()
        
        # Estado inicial
        if initial_state is None:
            initial_state = list(range(self.n_cities))
            random.shuffle(initial_state)
        
        super().__init__(tuple(initial_state))
    
    def _compute_distances(self) -> np.ndarray:
        """Calcula matriz de distâncias euclidianas."""
        n = len(self.cities)
        distances = np.zeros((n, n))
        for i in range(n):
            for j in range(i+1, n):
                dist = np.linalg.norm(self.cities[i] - self.cities[j])
                distances[i, j] = distances[j, i] = dist
        return distances
    
    def actions(self, state):
        """
        Ações: gerar vizinhos via 2-opt.
        Retorna lista de índices (i, j) para trocar.
        """
        n = len(state)
        return [(i, j) for i in range(n) for j in range(i+2, n)]
    
    def result(self, state, action):
        """
        Aplicar ação 2-opt: reverter segmento entre i e j.
        """
        state = list(state)
        i, j = action
        state[i:j+1] = reversed(state[i:j+1])
        return tuple(state)
    
    def value(self, state) -> float:
        """
        Valor heurístico: inverso da distância (para maximizar = minimizar distância).
        Usado por algoritmos de otimização (SA, AG).
        """
        cost = self._calculate_cost(state)
        return -cost  # Negativo porque queremos minimizar
    
    def _calculate_cost(self, state) -> float:
        """Calcula distância total de um tour."""
        cost = 0.0
        for i in range(len(state)):
            from_city = state[i]
            to_city = state[(i + 1) % len(state)]
            cost += self.distances[from_city, to_city]
        return cost
    
    def goal_test(self, state):
        """
        TSP é problema de otimização contínua, não tem 'objetivo' específico.
        Usamos value() em vez disso.
        """
        return False


# Criar instância de teste
np.random.seed(42)
cities = np.random.rand(10, 2) * 100
tsp_problem = TSPProblem(cities)

print(f"✓ TSPProblem criado")
print(f"  Cidades: {tsp_problem.n_cities}")
print(f"  Estado inicial: {tsp_problem.initial}")
print(f"  Custo inicial: {tsp_problem._calculate_cost(tsp_problem.initial):.2f}")

## 2. Simulated Annealing do AIMA

Função `simulated_annealing(problem, schedule)` do AIMA:
- Trabalha com qualquer `Problem`
- Usa `value()` para avaliar soluções
- `schedule` controla a temperatura

In [ ]:
# Importar simulated_annealing do AIMA (se disponível)
try:
    from aima.search import simulated_annealing
    print("✓ simulated_annealing importado de AIMA")
    use_aima_sa = True
except ImportError:
    print("⚠ simulated_annealing não encontrado, usar implementação manual")
    use_aima_sa = False


def schedule(t):
    """
    Agenda de temperatura para Simulated Annealing.
    T(t) = T0 * exp(-c*t)
    """
    return 1000.0 * math.exp(-0.005 * t)


if use_aima_sa:
    # Usar AIMA
    print("\nExecutando Simulated Annealing (AIMA)...")
    solution_aima = simulated_annealing(tsp_problem, schedule)
    cost_aima = tsp_problem._calculate_cost(solution_aima)
    print(f"Solução encontrada: {solution_aima}")
    print(f"Custo: {cost_aima:.2f}")
else:
    print("AIMA SA não disponível")
    solution_aima = None

## 3. Genetic Algorithm do AIMA

Função `genetic_algorithm(population, fitness_fn, gene_pool, f_thres, ...)` do AIMA:
- Seleção por roleta
- Crossover e mutação
- Parada quando atinge threshold de fitness

In [ ]:
try:
    from aima.search import genetic_algorithm, init_population
    print("✓ genetic_algorithm e init_population importados de AIMA")
    use_aima_ga = True
except ImportError:
    print("⚠ genetic_algorithm não encontrado")
    use_aima_ga = False


if use_aima_ga:
    # Gene pool para TSP: índices de cidades
    gene_pool = list(range(tsp_problem.n_cities))
    
    # Fitness function: inverso do custo
    def fitness_tsp(individual):
        cost = tsp_problem._calculate_cost(individual)
        return 1.0 / (1.0 + cost)
    
    print("\nExecutando Genetic Algorithm (AIMA)...")
    
    # Inicializar população
    population = init_population(pop_size=30, gene_pool=gene_pool, n=tsp_problem.n_cities)
    
    # Rodar AG
    solution_ga = genetic_algorithm(
        population,
        fitness_fn=fitness_tsp,
        gene_pool=gene_pool,
        f_thres=1e6,  # Alto (nunca atinge, para por iterações)
        ngen=100,
        pmut=0.1
    )
    
    cost_ga = tsp_problem._calculate_cost(solution_ga)
    print(f"Solução encontrada: {solution_ga}")
    print(f"Custo: {cost_ga:.2f}")
else:
    print("AIMA GA não disponível")
    solution_ga = None

## 4. Comparação: Nossa implementação vs AIMA

Ambas devem convergir para soluções similares.

In [ ]:
print("\n" + "="*70)
print("COMPARACAO: AIMA vs Nossa implementacao")
print("="*70)

# Carregar nossa implementação anterior
import sys
sys.path.insert(0, '.')
# (código da nossa implementação estaria aqui)

if solution_aima is not None:
    print(f"\nAIMA Simulated Annealing:")
    print(f"  Solução: {solution_aima}")
    print(f"  Custo: {cost_aima:.2f}")

if solution_ga is not None:
    print(f"\nAIMA Genetic Algorithm:")
    print(f"  Solução: {solution_ga}")
    print(f"  Custo: {cost_ga:.2f}")

if solution_aima and solution_ga:
    if cost_aima < cost_ga:
        diff = ((cost_ga - cost_aima) / cost_aima) * 100
        print(f"\n🏆 Vencedor: SA ({diff:.1f}% melhor)")
    else:
        diff = ((cost_aima - cost_ga) / cost_ga) * 100
        print(f"\n🏆 Vencedor: AG ({diff:.1f}% melhor)")

## 5. Explorar a estrutura do AIMA

Entender o que está disponível na biblioteca.

In [ ]:
import aima.search as search_module

print("="*70)
print("ALGORITMOS DISPONIVEIS EM aima.search")
print("="*70)

# Listar classes e funções principais
items = dir(search_module)
algorithms = [item for item in items if not item.startswith('_')]

print(f"\nTotal: {len(algorithms)} items")

# Categorizar
classes = [item for item in algorithms if item[0].isupper()]
functions = [item for item in algorithms if item[0].islower() and callable(getattr(search_module, item))]

print(f"\nClasses ({len(classes)}):")
for cls in sorted(classes)[:15]:
    print(f"  - {cls}")

print(f"\nFuncoes principais ({len(functions)}):")
for func in sorted(functions)[:15]:
    print(f"  - {func}")

## 6. Usa de outros algoritmos do AIMA para TSP

In [ ]:
# Hill climbing (também usa value())
try:
    from aima.search import hill_climbing
    print("\nExecutando Hill Climbing (AIMA)...")
    solution_hc = hill_climbing(tsp_problem)
    cost_hc = tsp_problem._calculate_cost(solution_hc)
    print(f"  Solução: {solution_hc}")
    print(f"  Custo: {cost_hc:.2f}")
    print(f"  (Hill climbing tende a ficar em ótimos locais)")
except ImportError:
    print("Hill climbing não disponível")

## 7. Visualização

In [ ]:
def plot_tour_aima(cities, tour, title, cost):
    """Plota tour do TSP."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plotar cidades
    ax.scatter(cities[:, 0], cities[:, 1], c='red', s=100, zorder=3)
    
    # Plotar tour
    tour_list = list(tour)
    tour_points = cities[tour_list + [tour_list[0]]]
    ax.plot(tour_points[:, 0], tour_points[:, 1], 'b-', linewidth=2, zorder=1)
    
    # Anotações
    for i, (x, y) in enumerate(cities):
        ax.annotate(str(i), (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(f"{title}\n(Custo: {cost:.2f})")
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    plt.tight_layout()
    return fig


if solution_aima is not None:
    fig1 = plot_tour_aima(cities, solution_aima, "AIMA: Simulated Annealing", cost_aima)
    plt.show()

if solution_ga is not None:
    fig2 = plot_tour_aima(cities, solution_ga, "AIMA: Genetic Algorithm", cost_ga)
    plt.show()

## 8. Próximos passos

- Integrar nosso benchmark com AIMA
- Testar em instâncias maiores
- Comparar desempenho
- Usar em Tarefa 1 final

In [ ]:
print("\n✓ Notebook AIMA + TSP concluído")
print("\nReferências:")
print("  - https://github.com/aimacode/aima-python")
print("  - Russell & Norvig (2020) — Artificial Intelligence (4ª ed.)")
print("  - Documentação: https://aimacode.github.io/aima-python/")